# Malayalam story with expressions: Indic Parler-TTS test for Mixlumo

Reads the **Theeppori the dragon** story aloud with per-line emotions, character voices, a dragon voice effect,
pauses and laughs, using **AI4Bharat Indic Parler-TTS** (Apache-2.0). Nothing is simulated.

**Before running (one time):**
1. Signed in at huggingface.co, open **https://huggingface.co/ai4bharat/indic-parler-tts** and accept the access
   conditions (screenshot them for the licence check).
2. Kaggle **Add-ons → Secrets**: `HF_TOKEN` (a Hugging Face **Read** token) switched **on** for this notebook.
3. Use a **fresh session** (new notebook, or **Run → Factory reset**), **GPU T4 x2**, **Internet On**, then **Run All**.
4. At the end download **`parler_story_eval.zip`**. The full story is **`story_full.wav`** inside it.

Measured too: does emotion really change the voice (pitch/energy/speed vs a neutral reading), GPU memory, speed,
whether it fits a **4 GB GPU** (GTX 1650), and CPU-only speed.

In [ ]:
# 1. Environment
import os, subprocess, sys, json, time, platform, shutil
OUT = "/kaggle/working/parler_story_eval" if os.path.isdir("/kaggle/working") else os.path.abspath("parler_story_eval")
os.makedirs(f"{OUT}/lines", exist_ok=True)
RESULTS = {"started": time.strftime("%Y-%m-%d %H:%M:%S"), "python": platform.python_version(), "steps": {}}
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() if shutil.which("nvidia-smi") else ""
RESULTS["gpu"] = smi
print("Python", platform.python_version(), "| disk free GB:", shutil.disk_usage("/").free // 2**30)
if not smi:
    raise SystemExit("NO GPU ATTACHED. Kaggle: Session options > Accelerator > GPU T4 x2.")
print("GPU:", smi)

In [ ]:
# 2. Install Parler-TTS (pinned commit), keeping the platform's PyTorch unchanged
if "transformers" in sys.modules:
    raise SystemExit("transformers is already loaded: Run > Restart & clear cell outputs, then Run all.")
from importlib import metadata
PARLER_COMMIT = "d108732"
LOCK = {}
for pkg in ("torch", "torchaudio"):
    try:
        LOCK[pkg] = metadata.version(pkg)
    except metadata.PackageNotFoundError:
        pass
mm = lambda v: ".".join(v.split("+")[0].split(".")[:2])
installed = {d.metadata["Name"].lower() for d in metadata.distributions()}
leftovers = [p for p in ("maya3", "f5-tts", "f5_tts") if p in installed]
if leftovers or ("torch" in LOCK and "torchaudio" in LOCK and mm(LOCK["torch"]) != mm(LOCK["torchaudio"])):
    raise SystemExit(f"Leftovers from an earlier run ({leftovers}, torch {LOCK.get('torch')}, torchaudio "
                     f"{LOCK.get('torchaudio')}). Use a NEW notebook or Run > Factory reset, then Run all.")
open(f"{OUT}/constraints.txt", "w").write("\n".join(f"{k}=={v}" for k, v in LOCK.items()) + "\n")
print("keeping", LOCK)

CHECK = ("import torch, transformers; from parler_tts import ParlerTTSForConditionalGeneration; "
         "print('imports OK: torch', torch.__version__, '| transformers', transformers.__version__)")
def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args, "-c", f"{OUT}/constraints.txt"],
                       capture_output=True, text=True)
    return r.returncode, (r.stdout[-1500:] + r.stderr[-3000:])
def check():
    tv = subprocess.run([sys.executable, "-c", "import torchvision"], capture_output=True, text=True)
    if tv.returncode != 0 and "No module named" not in tv.stderr:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"], capture_output=True)
        print("removed a torchvision that doesn't match this PyTorch (not needed here)")
    return subprocess.run([sys.executable, "-c", CHECK], capture_output=True, text=True)

t = time.time()
rc, log = pip(f"git+https://github.com/huggingface/parler-tts@{PARLER_COMMIT}")
c = check()
route = "normal"
if rc != 0 or c.returncode != 0:
    print("normal install didn't work on this Python; trying the minimal install\n", log[-1500:], c.stderr[-1500:])
    route = "minimal"
    rc1, l1 = pip("--no-deps", f"git+https://github.com/huggingface/parler-tts@{PARLER_COMMIT}",
                  "descript-audio-codec", "git+https://github.com/descriptinc/audiotools")
    rc2, l2 = pip("transformers==4.46.1", "sentencepiece", "protobuf", "argbind", "einops", "julius", "flatten-dict",
                  "ffmpy", "rich", "pyloudnorm", "importlib-resources", "randomname", "markdown2", "pystoi", "torch-stoi",
                  "librosa", "scipy", "soundfile", "matplotlib", "tensorboard", "tqdm")
    log = l1 + l2
    c = check()
print(c.stdout, c.stderr[-3000:])
RESULTS["steps"]["install"] = {"route": route, "ok": c.returncode == 0, "seconds": round(time.time() - t, 1),
                               "output": (c.stdout + c.stderr)[-1500:]}
assert c.returncode == 0, "Parler-TTS imports fail: see the error above"

In [ ]:
# 3. Hugging Face access (gated model). The token comes from Kaggle/Colab secrets and is never printed.
def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        pass
    return os.environ.get("HF_TOKEN")
tok = hf_token()
if not tok:
    raise SystemExit("No HF_TOKEN secret found. Kaggle: Add-ons > Secrets > HF_TOKEN, switch it ON for this notebook.")
os.environ["HF_TOKEN"] = tok
from huggingface_hub import HfApi
MODEL = "ai4bharat/indic-parler-tts"
try:
    info = HfApi().model_info(MODEL, token=tok)
    RESULTS["model_info"] = {"sha": info.sha, "license": (info.card_data or {}).get("license") if info.card_data else None,
                             "files": sorted(s.rfilename for s in info.siblings)}
    print("access OK | revision", info.sha, "| licence tag", RESULTS["model_info"]["license"])
except Exception as e:
    raise SystemExit(f"Cannot access {MODEL} ({type(e).__name__}). Open its page while signed in, accept the "
                     "conditions, check the token is a Read token, then Run all again.")

In [ ]:
# 4. The story and the script parser (Mixlumo's story_script.py, unit-tested in the Mixlumo repo)
open("story_script.py", "w", encoding="utf-8").write('# Copyright (c) 2026 Achu Prasad. All rights reserved. See LICENSE.\n"""Malayalam story scripts with inline expression tags -> a list of performance steps.\n\nInput is plain text like:\n\n    കേരളത്തിലെ ... ഗ്രാമമായ പുഞ്ചക്കരയിൽ, ...           (narration)\n    [excited] ഒരു കൂറ്റൻ ഡ്രാഗൺ ആയിരുന്നു!               (narration with an emotion)\n    [angrily] രാമേട്ടൻ: "എന്താടാ ഇത്? ..."               (a named character speaking)\n    [dragon voice] "ഗ്ര്ര്ര്ര്... എനിക്ക് ഭക്ഷണം വേണം!"  (a voice effect)\n    [dramatic pause]                                      (an event on its own line)\n    [ giggles ]                                           (a non-verbal sound)\n\nEach step is one of:\n    {"kind": "speech", "text", "speaker", "emotion", "effect", "pause_after"}\n    {"kind": "pause", "seconds"}\n    {"kind": "laugh", "style"}\n\nTags are never spoken. Unknown tags are reported in `warnings` and ignored. Emojis are removed. This module\nis model-independent: a TTS adapter turns `emotion` into whatever its model understands (a description for\nIndic Parler-TTS, an inline tag for Maya), and `effect` is applied afterwards with the existing audio engine.\n"""\nimport re\nimport unicodedata\n\n# tag (lowercase, spaces collapsed) -> (category, value)\nEMOTIONS = {\n    "neutral": "neutral", "calm": "calm", "happy": "happy", "happily": "happy", "excited": "excited",\n    "excitedly": "excited", "angry": "angry", "angrily": "angry", "sad": "sad", "sadly": "sad", "crying": "crying",\n    "fearful": "fearful", "scared": "fearful", "afraid": "fearful", "surprised": "surprised", "shocked": "surprised",\n    "confused": "confused", "proudly": "proud", "proud": "proud", "confident": "confident", "deadpan": "deadpan",\n    "sarcastic": "sarcastic", "sarcastically": "sarcastic", "romantic": "romantic", "dramatically": "dramatic",\n    "dramatic": "dramatic", "whispers": "whisper", "whispering": "whisper", "whisper": "whisper",\n}\nEFFECTS = {"dragon voice": "monster", "monster voice": "monster", "robot voice": "robot", "echo": "echo",\n           "radio voice": "radio", "phone voice": "radio"}\nPAUSES = {"dramatic pause": 1.2, "pause": 0.7, "long pause": 1.5, "short pause": 0.35, "beat": 0.5}\nLAUGHS = {"giggles": "giggle", "giggle": "giggle", "laughs": "laugh", "laughing": "laugh", "laugh": "laugh",\n          "bursts into laughter": "big", "chuckles": "giggle"}\n\nTAG_RE = re.compile(r"\\[\\s*([^\\[\\]]{1,40}?)\\s*\\]")\n# a speaker label at the start of a line: up to 3 words, then \':\' or \':-\', not containing quotes or sentence marks\nSPEAKER_RE = re.compile(r"^\\s*([^\\s\\"“”:?!.,][^\\"“”:?!.]{0,40}?)\\s*:\\s*-?\\s*(?=[\\"“]|\\S)")\nQUOTE_RE = re.compile(r"^[\\"“](.*)[\\"”]$", re.S)\nMAX_LINE_CHARS = 600\n\n\ndef strip_emoji(s):\n    """Remove emoji and other pictographic symbols; keep Malayalam (incl. ZWJ/ZWNJ) and punctuation."""\n    out = []\n    for ch in s:\n        if ch in ("\u200c", "\u200d"):            # zero-width (non-)joiner are part of Malayalam spelling\n            out.append(ch)\n            continue\n        cat = unicodedata.category(ch)\n        if cat == "So" or ch in ("️", "︎") or 0x1F000 <= ord(ch) <= 0x1FAFF:\n            continue\n        out.append(ch)\n    return "".join(out)\n\n\ndef _norm_tag(t):\n    return " ".join(t.lower().replace("_", " ").split())\n\n\ndef parse(script, narrator="narrator"):\n    """Returns (steps, warnings). Speech steps carry the speaker label as written (e.g. \'രാമേട്ടൻ\') or\n    `narrator`; quoted lines without a label get speaker \'voice\' (an unnamed character)."""\n    if not isinstance(script, str):\n        raise ValueError("script must be text")\n    steps, warnings = [], []\n    text = unicodedata.normalize("NFC", script.replace("\\r\\n", "\\n"))\n    for lineno, raw in enumerate(text.split("\\n"), 1):\n        line = strip_emoji(raw).strip()\n        if not line:\n            continue\n        emotion, effect, events = None, None, []\n        for m in TAG_RE.finditer(line):\n            tag = _norm_tag(m.group(1))\n            if tag in EMOTIONS:\n                emotion = EMOTIONS[tag]\n            elif tag in EFFECTS:\n                effect = EFFECTS[tag]\n            elif tag in PAUSES:\n                events.append({"kind": "pause", "seconds": PAUSES[tag]})\n            elif tag in LAUGHS:\n                events.append({"kind": "laugh", "style": LAUGHS[tag]})\n            else:\n                warnings.append(f"line {lineno}: unknown tag [{m.group(1).strip()}] ignored")\n        body = " ".join(TAG_RE.sub(" ", line).split())\n        if not body:                       # a line that is only tags: pauses / laughs\n            steps.extend(events)\n            if not events and (emotion or effect):\n                warnings.append(f"line {lineno}: tag without text ignored")\n            continue\n        if len(body) > MAX_LINE_CHARS:\n            warnings.append(f"line {lineno}: longer than {MAX_LINE_CHARS} characters; split it for best results")\n        speaker = narrator\n        m = SPEAKER_RE.match(body)\n        if m and len(m.group(1).split()) <= 3 and body[m.end():].strip():\n            speaker, body = m.group(1).strip(), body[m.end():].strip()\n        q = QUOTE_RE.match(body)\n        if q:\n            body = q.group(1).strip()\n            if speaker == narrator:\n                speaker = "voice"\n        pause_after = 0.6 if body.endswith(("...", "…")) else 0.35\n        steps.append({"kind": "speech", "text": body, "speaker": speaker, "emotion": emotion or "neutral",\n                      "effect": effect, "pause_after": pause_after, "line": lineno})\n        steps.extend(events)                # tags like "[giggles]" written after the text on the same line\n    return steps, warnings\n\n\n# Malayalam number words for small numbers, so digits are not skipped or read letter by letter.\n_ML_NUM = {0: "പൂജ്യം", 1: "ഒന്ന്", 2: "രണ്ട്", 3: "മൂന്ന്", 4: "നാല്", 5: "അഞ്ച്", 6: "ആറ്", 7: "ഏഴ്",\n           8: "എട്ട്", 9: "ഒമ്പത്", 10: "പത്ത്", 20: "ഇരുപത്", 50: "അമ്പത്", 100: "നൂറ്"}\n\n\ndef numbers_to_words(text, warnings=None):\n    """Replace standalone digits that have a known Malayalam word; others are left as they are and reported."""\n    def rep(m):\n        n = int(m.group(0))\n        if n in _ML_NUM:\n            return _ML_NUM[n]\n        if warnings is not None:\n            warnings.append(f"number {n} left as digits: write it in words for reliable speech")\n        return m.group(0)\n    return re.sub(r"(?<![\\d.])\\d+(?![\\d.])", rep, text)\n')
import importlib, story_script as ss
importlib.reload(ss)
STORY = 'കേരളത്തിലെ മനോഹരമായ ഒരു കൊച്ചു ഗ്രാമമായ പുഞ്ചക്കരയിൽ, ഒരു ദിവസം രാവിലെ ആകാശത്തുനിന്ന് എന്തോ വലിയൊരു സാധനം താഴേക്ക് പറന്നുവന്നു.\n\n[dramatically] അത് ഒരു ഹെലികോപ്റ്ററായിരുന്നില്ല... വിമാനവുമായിരുന്നില്ല...\n\n[excited] ഒരു കൂറ്റൻ ഡ്രാഗൺ ആയിരുന്നു!\n\nഅവന്റെ പേര് തീപ്പൊരി!\n\nപക്ഷേ ഒരു ചെറിയ പ്രശ്നമുണ്ടായിരുന്നു...\n\nഅവന് തീ തുപ്പാൻ അറിയാം. പക്ഷേ മലയാളം അറിയില്ല!\n\n[confused] ഗ്രാമത്തിലെത്തിയ ഉടനെ അവൻ നേരെ പോയത് രാമേട്ടന്റെ ചായക്കടയിലേക്കാണ്.\n\n[dragon voice] "ഗ്ര്ര്ര്ര്... എനിക്ക് ഭക്ഷണം വേണം!"\n\n[angrily] രാമേട്ടൻ: "എന്താടാ ഇത്? രാവിലെ തന്നെ ഗ്യാസ് സിലിണ്ടർ പൊട്ടിയ പോലെ ഒരു ശബ്ദം!"\n\nഡ്രാഗൺ വിശന്നിട്ട് വീണ്ടും അലറി.\n\nരാമേട്ടൻ ഒരു ചായയും രണ്ട് പഴംപൊരിയും മുന്നിൽ വെച്ചു.\n\n[excited] ഡ്രാഗൺ അത് ഒറ്റയടിക്ക് അകത്താക്കി!\n\nപക്ഷേ അടുത്ത നിമിഷം...\n\n[shocked] അവന്റെ വായിൽനിന്ന് തീ പുറത്തേക്ക് വന്നു!\n\nചായക്കടയിലെ ഗ്യാസ് സ്റ്റൗ പോലും നാണിച്ചുപോയി!\n\n[deadpan] രാമേട്ടൻ: "എടാ മോനേ... നീ ഇവിടെ വന്നത് ചായ കുടിക്കാനാണോ, അതോ എന്റെ കട കത്തിക്കാനാണോ?"\n\n[ giggles ]\n\nഅപ്പോഴേക്കും നാട്ടുകാർ ഓടിക്കൂടി.\n\nഒരാൾ ചോദിച്ചു:\n\n"ഇതിനെ പിടിക്കാൻ വനംവകുപ്പിനെ വിളിക്കണോ?"\n\nമറ്റൊരാൾ പറഞ്ഞു:\n\n"വനംവകുപ്പ് വേണ്ട... ആദ്യം ഫയർഫോഴ്സിനെ വിളിക്ക്!"\n\nപക്ഷേ തീപ്പൊരിക്ക് നാട്ടുകാരെ ഇഷ്ടമായി.\n\nഅവൻ ദിവസവും ചായക്കടയിൽ വരാൻ തുടങ്ങി.\n\nഒരു ദിവസം രാമേട്ടൻ ചോദിച്ചു:\n\n"എടാ തീപ്പൊരീ, നിനക്ക് ജോലിയൊന്നും വേണ്ടേ?"\n\n[proudly] ഡ്രാഗൺ നെഞ്ചുവിരിച്ച് നിന്നു.\n\nഅങ്ങനെ അവന് ഗ്രാമത്തിൽ ഒരു ജോലി കിട്ടി.\n\nപഞ്ചായത്തിന്റെ പുതിയ തേങ്ങയിടൽ മെഷീൻ!\n\n[ laughing ]\n\nഇപ്പോൾ പുഞ്ചക്കരയിൽ തേങ്ങയിടാൻ ആരും കയറേണ്ടതില്ല.\n\nതീപ്പൊരി ഒന്ന് ശ്വാസം വിട്ടാൽ മതി...\n\nതേങ്ങയും വീഴും, ചിലപ്പോൾ തെങ്ങും വീഴും!\n\n[whispers] പക്ഷേ ഇന്നും ഒരു രഹസ്യമുണ്ട്...\n\nതീപ്പൊരിക്ക് ഏറ്റവും പേടിയുള്ളത് നാട്ടുകാരെയല്ല...\n\nരാമേട്ടന്റെ കടയിലെ 10 രൂപയുടെ കടം ചോദിക്കുന്ന ആ പഴയ കണക്കുപുസ്തകത്തെയാണ്!\n\n[dramatic pause]\n\nകാരണം...\n\nഡ്രാഗണിന് ലോകം മുഴുവൻ കത്തിക്കാൻ കഴിയും...\n\nപക്ഷേ രാമേട്ടന്റെ കടം വീട്ടാൻ മാത്രം കഴിയില്ല!\n\n[bursts into laughter] 🤣\n'
STEPS, WARN = ss.parse(STORY)
print(len(STEPS), "steps |", sum(s["kind"] == "speech" for s in STEPS), "spoken lines | warnings:", WARN or "none")
for i, s in enumerate(STEPS):
    if s["kind"] == "speech":
        print(f"{i:2d} {s['speaker'][:10]:10s} {s['emotion']:9s} {s['effect'] or '':8s} {s['text'][:60]}")
    else:
        print(f"{i:2d} [{s['kind']}: {s.get('style') or s.get('seconds')}]")
RESULTS["script"] = {"steps": STEPS, "warnings": WARN}

### 5. Voices and emotions

Casting (Indic Parler-TTS' recommended Malayalam voices): narrator **Anjali**, Ramettan **Harish**, other quoted
voices **Anju**, the dragon **Harish** with a monster effect (pitch down, growl, echo) added afterwards by
Mixlumo-style audio processing. Each emotion becomes a plain-language voice description, which is how this model
is controlled. Edit `CAST` / `EMOTION` and re-run from here to try other choices.

In [ ]:
import torch, numpy as np, soundfile as sf
from transformers import AutoTokenizer, set_seed
from parler_tts import ParlerTTSForConditionalGeneration

CAST = {"narrator": "Anjali", "രാമേട്ടൻ": "Harish", "voice": "Anju"}
MONSTER_VOICE = "Harish"
EMOTION = {
    "neutral":   "narrates a story in an expressive, engaging storytelling voice at a moderate pace",
    "dramatic":  "narrates dramatically and suspensefully, slowly, with long pauses and an expressive voice",
    "excited":   "speaks in an excited, happy and energetic tone, slightly fast, with a high pitch",
    "happy":     "speaks in a happy, cheerful tone",
    "confused":  "speaks in a confused, puzzled and hesitant tone at a slow pace",
    "angry":     "speaks in an angry tone, loud and forceful, with a fast pace",
    "surprised": "speaks in a shocked, surprised tone with a high pitch",
    "deadpan":   "speaks in a flat, monotone, deadpan voice with a slow pace and a low pitch",
    "proud":     "speaks in a proud, confident tone",
    "whisper":   "whispers very softly and quietly, in a hushed, breathy voice at a slow pace",
    "sad":       "speaks in a sad tone, slowly, with a low pitch",
    "fearful":   "speaks in a fearful, scared tone",
    "calm":      "speaks in a calm, gentle tone",
    "sarcastic": "speaks in a sarcastic tone",
    "crying":    "speaks in a sad, tearful, crying voice",
    "romantic":  "speaks in a soft, warm and romantic tone",
    "confident": "speaks in a confident tone",
}
QUALITY = "The recording is of very high quality, with the speaker's voice sounding clear and very close up."
def describe(name, emotion, monster=False):
    if monster:
        return f"{name} speaks in a deep, slow, growling, menacing voice with a very low pitch. {QUALITY}"
    return f"{name} {EMOTION.get(emotion, EMOTION['neutral'])}. {QUALITY}"

torch.cuda.reset_peak_memory_stats(); t = time.time()
model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL, token=tok).to("cuda")
tokenizer = AutoTokenizer.from_pretrained(MODEL, token=tok)
desc_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path, token=tok)
SR = model.config.sampling_rate
RESULTS["load"] = {"seconds": round(time.time() - t, 1), "sampling_rate": SR,
                   "params_millions": round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
                   "gpu_gb_after_load": round(torch.cuda.memory_allocated() / 2**30, 2)}
print(RESULTS["load"])

def say(text, description, seed=0):
    set_seed(seed)
    d = desc_tokenizer(description, return_tensors="pt").to("cuda")
    p = tokenizer(text, return_tensors="pt").to("cuda")
    g = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                       prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)
    return g.cpu().numpy().squeeze().astype(np.float32)

In [ ]:
# 6. Audio helpers (the Mixlumo app does these with its existing audio engine)
import librosa
def stats(w):
    w = np.asarray(w, np.float32)
    return {"audio_s": round(len(w) / SR, 2), "peak": round(float(np.abs(w).max()), 3) if len(w) else 0,
            "rms_dbfs": round(float(20 * np.log10(np.sqrt(np.mean(w ** 2)) + 1e-9)), 1) if len(w) else None,
            "finite": bool(np.isfinite(w).all())}
def trim(w, db=40):
    yt, _ = librosa.effects.trim(w, top_db=db)
    return yt
def level(w, target_db=-20.0):
    rms = np.sqrt(np.mean(w ** 2)) + 1e-9
    w = w * (10 ** (target_db / 20) / rms)
    peak = np.abs(w).max()
    return w * (0.95 / peak) if peak > 0.95 else w
def monster(w):
    y = librosa.effects.pitch_shift(w, sr=SR, n_steps=-5)                 # deeper
    y = np.tanh(2.2 * y) / np.tanh(2.2)                                   # growl
    d = int(0.12 * SR); e = np.zeros(len(y) + 3 * d, np.float32); e[:len(y)] += y
    for k, g in ((1, 0.35), (2, 0.15), (3, 0.06)):                         # cave echo
        e[k * d:k * d + len(y)] += g * y
    return e
def silence(s):
    return np.zeros(int(s * SR), np.float32)

In [ ]:
# 7. Perform the story, line by line
LINES, PIECES = [], []
for i, s in enumerate(STEPS):
    rec = {"step": i, **{k: v for k, v in s.items() if k != "line"}}
    try:
        if s["kind"] == "pause":
            PIECES.append(silence(s["seconds"])); rec["audio_s"] = s["seconds"]
        elif s["kind"] == "laugh":           # experimental: the model "reading" laughter
            text = {"giggle": "ഹി ഹി ഹി!", "laugh": "ഹ ഹ ഹ ഹ!", "big": "ഹ ഹ ഹ ഹ ഹ ഹ ഹ ഹ!"}[s["style"]]
            desc = f"{CAST['narrator']} laughs out loud joyfully and happily. {QUALITY}"
            torch.cuda.synchronize(); t = time.time(); w = say(text, desc, seed=i); torch.cuda.synchronize()
            rec.update(text=text, description=desc, gen_seconds=round(time.time() - t, 2))
            w = level(trim(w), -22); PIECES += [w, silence(0.4)]; rec.update(stats(w))
            sf.write(f"{OUT}/lines/{i:02d}_laugh_{s['style']}.wav", w, SR)
        else:
            text = ss.numbers_to_words(s["text"], WARN)
            is_monster = s.get("effect") == "monster"
            name = MONSTER_VOICE if is_monster else CAST.get(s["speaker"], CAST["voice"])
            desc = describe(name, s["emotion"], monster=is_monster)
            torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t = time.time()
            w = say(text, desc, seed=i)
            torch.cuda.synchronize(); dt = time.time() - t
            rec.update(text=text, voice=name, description=desc, gen_seconds=round(dt, 2),
                       rtf=round(dt / max(len(w) / SR, 1e-6), 3), peak_gpu_gb=round(torch.cuda.max_memory_allocated() / 2**30, 2))
            w = trim(w)
            if is_monster:
                w = monster(w)
            w = level(w, -27 if s["emotion"] == "whisper" else -20)
            PIECES += [w, silence(s["pause_after"])]; rec.update(stats(w))
            sf.write(f"{OUT}/lines/{i:02d}_{s['emotion']}{'_monster' if is_monster else ''}.wav", w, SR)
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {e}"[:600]
    LINES.append(rec)
    print(i, {k: rec.get(k) for k in ("kind", "voice", "emotion", "audio_s", "gen_seconds", "rtf", "error")})

story = np.concatenate([silence(0.3)] + PIECES + [silence(0.5)])
sf.write(f"{OUT}/story_full.wav", story, SR)
RESULTS["lines"] = LINES
RESULTS["story"] = {"seconds": round(len(story) / SR, 1), "sampling_rate": SR,
                    "generation_seconds": round(sum(l.get("gen_seconds", 0) for l in LINES), 1),
                    "errors": [l["step"] for l in LINES if "error" in l], "warnings": WARN}
json.dump(RESULTS, open(f"{OUT}/results.json", "w", encoding="utf-8"), indent=1, default=str, ensure_ascii=False)
print(RESULTS["story"])

In [ ]:
# 8. Does the emotion description really change the voice? Same sentence, neutral vs emotion, same voice and seed.
#    Measured: median pitch (Hz), loudness before levelling (dBFS) and speaking time.
def measure(w):
    f0, vflag, _ = librosa.pyin(w, fmin=70, fmax=500, sr=SR, frame_length=2048)
    f0 = f0[~np.isnan(f0)]
    return {"pitch_hz": round(float(np.median(f0)), 1) if len(f0) else None,
            "loudness_dbfs": round(float(20 * np.log10(np.sqrt(np.mean(w ** 2)) + 1e-9)), 1),
            "seconds": round(len(w) / SR, 2)}
AB_TEXT = "എനിക്ക് ഇത് ഒട്ടും വിശ്വസിക്കാൻ കഴിയുന്നില്ല, ഇത് ശരിക്കും സംഭവിച്ചതാണോ?"   # I can't believe this at all, did it really happen?
AB = []
for voice in ("Anjali", "Harish"):
    base = measure(trim(say(AB_TEXT, describe(voice, "neutral"), seed=7)))
    for emo in ("excited", "angry", "sad", "whisper", "deadpan", "surprised"):
        w = trim(say(AB_TEXT, describe(voice, emo), seed=7))
        sf.write(f"{OUT}/lines/ab_{voice}_{emo}.wav", level(w, -27 if emo == "whisper" else -20), SR)
        AB.append({"voice": voice, "emotion": emo, "neutral": base, "emotion_result": measure(w)})
        print(AB[-1])
RESULTS["emotion_ab"] = AB
json.dump(RESULTS, open(f"{OUT}/results.json", "w", encoding="utf-8"), indent=1, default=str, ensure_ascii=False)

In [ ]:
# 9. Fits a 4 GB GPU (GTX 1650)? A separate process limited to 3.3 GB, in float16 and float32.
LIMIT = "\n".join([
    "import json, os, sys, time, numpy as np, torch",
    "from transformers import AutoTokenizer",
    "from parler_tts import ParlerTTSForConditionalGeneration",
    "half = sys.argv[1] == '1'; M = 'ai4bharat/indic-parler-tts'; tok = os.environ['HF_TOKEN']",
    "torch.cuda.set_per_process_memory_fraction(3.3 * 2**30 / torch.cuda.get_device_properties(0).total_memory)",
    "rec = {'limit_gb': 3.3, 'half': half}",
    "try:",
    "    m = ParlerTTSForConditionalGeneration.from_pretrained(M, token=tok, torch_dtype=torch.float16 if half else torch.float32).to('cuda')",
    "    tk = AutoTokenizer.from_pretrained(M, token=tok); dt_ = AutoTokenizer.from_pretrained(m.config.text_encoder._name_or_path, token=tok)",
    "    d = dt_(sys.argv[2], return_tensors='pt').to('cuda'); p = tk(sys.argv[3], return_tensors='pt').to('cuda')",
    "    t = time.time()",
    "    g = m.generate(input_ids=d.input_ids, attention_mask=d.attention_mask, prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)",
    "    torch.cuda.synchronize(); w = g.float().cpu().numpy().squeeze()",
    "    rec.update(ok=bool(np.isfinite(w).all() and len(w) > m.config.sampling_rate), audio_s=round(len(w) / m.config.sampling_rate, 2),",
    "               gen_seconds=round(time.time() - t, 2), peak_gb=round(torch.cuda.max_memory_allocated() / 2**30, 2))",
    "except BaseException as e:",
    "    rec.update(ok=False, error=(type(e).__name__ + ': ' + str(e))[:600])",
    "print('RESULT ' + json.dumps(rec))",
])
open(f"{OUT}/limit_test.py", "w").write(LIMIT)
RESULTS["fits_4gb"] = []
long_line = next(s["text"] for s in STEPS if s["kind"] == "speech" and len(s["text"]) > 100)
for half in ("1", "0"):
    p = subprocess.run([sys.executable, f"{OUT}/limit_test.py", half, describe("Anjali", "neutral"), long_line],
                       capture_output=True, text=True, timeout=1800, env=dict(os.environ))
    line = next((l for l in p.stdout.splitlines() if l.startswith("RESULT ")), None)
    r = json.loads(line[7:]) if line else {"half": half == "1", "ok": False, "error": "process died", "stderr": p.stderr[-1200:]}
    RESULTS["fits_4gb"].append(r); print(r)
json.dump(RESULTS, open(f"{OUT}/results.json", "w", encoding="utf-8"), indent=1, default=str, ensure_ascii=False)

In [ ]:
# 10. CPU only: speed for one medium line (Kaggle's CPU is slower than most desktops)
CPU = "\n".join([
    "import json, os, sys, time, numpy as np, torch",
    "from transformers import AutoTokenizer",
    "from parler_tts import ParlerTTSForConditionalGeneration",
    "M = 'ai4bharat/indic-parler-tts'; tok = os.environ['HF_TOKEN']; rec = {'threads': torch.get_num_threads()}",
    "try:",
    "    m = ParlerTTSForConditionalGeneration.from_pretrained(M, token=tok)",
    "    tk = AutoTokenizer.from_pretrained(M, token=tok); dt_ = AutoTokenizer.from_pretrained(m.config.text_encoder._name_or_path, token=tok)",
    "    d = dt_(sys.argv[1], return_tensors='pt'); p = tk(sys.argv[2], return_tensors='pt')",
    "    t = time.time(); g = m.generate(input_ids=d.input_ids, attention_mask=d.attention_mask, prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)",
    "    w = g.cpu().numpy().squeeze(); dt = time.time() - t",
    "    rec.update(audio_s=round(len(w) / m.config.sampling_rate, 2), gen_seconds=round(dt, 1), rtf=round(dt / max(len(w) / m.config.sampling_rate, 1e-6), 2))",
    "except BaseException as e:",
    "    rec['error'] = (type(e).__name__ + ': ' + str(e))[:600]",
    "print('RESULT ' + json.dumps(rec))",
])
open(f"{OUT}/cpu_test.py", "w").write(CPU)
mid = next(s["text"] for s in STEPS if s["kind"] == "speech" and 40 < len(s["text"]) < 80)
try:
    p = subprocess.run([sys.executable, f"{OUT}/cpu_test.py", describe("Anjali", "neutral"), mid], capture_output=True,
                       text=True, timeout=2400, env=dict(os.environ, CUDA_VISIBLE_DEVICES=""))
    line = next((l for l in p.stdout.splitlines() if l.startswith("RESULT ")), None)
    RESULTS["cpu"] = json.loads(line[7:]) if line else {"error": "process died", "stderr": p.stderr[-1200:]}
except subprocess.TimeoutExpired:
    RESULTS["cpu"] = {"error": "timeout after 40 min"}
print(RESULTS["cpu"])
json.dump(RESULTS, open(f"{OUT}/results.json", "w", encoding="utf-8"), indent=1, default=str, ensure_ascii=False)

In [ ]:
# 11. Summary + zip for download
ok = [l for l in LINES if "error" not in l and l["kind"] == "speech"]
RESULTS["summary"] = {
    "story_seconds": RESULTS["story"]["seconds"], "lines_ok": len(ok),
    "lines_failed": len(RESULTS["story"]["errors"]),
    "median_rtf_gpu": round(float(np.median([l["rtf"] for l in ok])), 3) if ok else None,
    "max_peak_gpu_gb": max((l.get("peak_gpu_gb", 0) for l in ok), default=None),
    "fits_4gb": [{k: r.get(k) for k in ("half", "ok", "peak_gb", "gen_seconds", "error")} for r in RESULTS.get("fits_4gb", [])],
    "cpu": RESULTS.get("cpu"),
}
RESULTS["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
json.dump(RESULTS, open(f"{OUT}/results.json", "w", encoding="utf-8"), indent=1, default=str, ensure_ascii=False)
print(json.dumps(RESULTS["summary"], indent=1, ensure_ascii=False))
shutil.make_archive(OUT, "zip", OUT)
print("Download:", OUT + ".zip", round(os.path.getsize(OUT + ".zip") / 2**20, 1), "MB  (story: story_full.wav)")